# Building Firmware for the Acadia Control System

A major benefit of the Acadia architecture is its ability to be easily reconfigured with firmware images defined in Python, rather than in pure VHDL. This guide will walk through the build process for a Python-defined image and explain the various utilities provided for doing so.

First, we'll import the necessary libraries. The firmware we want to build is written as a class that we can directly import:

In [1]:
from acadia.standardfirmware import StandardFirmware

No module named 'pyxrfclk'


The first step is to create a directory for storing the Vivado project that we'll generate. We then instantiate our firmware object with this path (if the directory doesn't exist, this instantiation will create it):

In [2]:
firmware = StandardFirmware("/home/billy/acadia-build-ps64")

The firmware may define a number of custom VHDL modules. We need to write these to a file that the Vivado project can then import:

In [3]:
firmware.write_hdl()

Then, we need to create a TCL script that will populate the HEDGEHOG logic with the relevant objects for this type of firmware:

In [4]:
firmware.write_hedgehog_tcl()

The final step before building is to generate a file containing any constraints needed for the HEDGEHOG logic:

In [5]:
firmware.write_hedgehog_constraints()

Then, we run the provided project creation script in a terminal to command Vivado to create the project in our example directory and configure it with the files we just wrote:

```
vivado -mode tcl -source /home/billy/acadia/logic/scripts/make_project.tcl -tclargs --project_dir /home/billy/acadia-build --origin_dir /home/billy/acadia/logic/src
```

If you're planning to simulate Acadia in a testbench, execute the following commands in the TCL console once the project assembly has completed:

```
create_bd_port -dir I debug_seq_nrst
create_bd_port -dir I debug_seq_run
delete_bd_objs [get_bd_nets seq_run_slice_Dout]
delete_bd_objs [get_bd_nets seq_rst_slice_Dout]
connect_bd_net [get_bd_ports debug_seq_nrst] [get_bd_pins xpm_cdc_seq_rst/src_rst]
connect_bd_net [get_bd_ports debug_seq_run] [get_bd_pins xpm_cdc_seq_run/src_rst]
validate_bd_design
save_bd_design
```